# NAIP and canopy height for the Elkinsville NE mapping module

Produces one four-band NAIP mosaic and one canopy height model per imagery date
over the AIS photo-interpretation module in Brown County, Indiana, all on a
single analysis grid, so that a stand can be measured the same way at every
date and differences between dates need no resampling.

Runtime: **Runtime > Change runtime type > T4 GPU**. The canopy height model is
PyTorch, so a CPU runtime will run but will take hours instead of minutes.

Steps: configure, install, mount Drive, fetch the conditioning rasters, fetch
NAIP per year, run inference per year, put every output on the analysis grid,
verify the alignment, export.

The heavy pieces are cached on Drive, so re-running after a disconnect skips
whatever is already done.

## 1. Configuration

In [1]:
from pathlib import Path

SITE = "SantaClara"

# Imagery dates. Planetary Computer carries this module from 2012 onward.
# 2012 and 2014 are 1.0 m; 2016 onward are 0.6 m.
YEARS = [2012, 2014, 2016, 2018, 2020, 2022]
SOURCE = "pc"                 # "pc" = Planetary Computer, "gee" = Earth Engine
EE_PROJECT = "dyce-biomass"   # only used when SOURCE == "gee"

# The analysis grid. Bounds are the module footprint snapped outward to a 3 m
# multiple, which is a whole number of pixels at both 0.6 m and 1.0 m so the
# two grids nest. Inference runs on a 150 m buffer of this, because the canopy
# height model blends chips and its outermost ~50 m is unreliable.
ANALYSIS_BOUNDS = (559425.0, 4323936.0, 564888.0, 4330917.0)   # EPSG:26916
INFER_BOUNDS = (559275.0, 4323786.0, 565038.0, 4331067.0)
STAC_BBOX_4326 = (-86.3125, 39.0625, -86.25, 39.1250)
DST_CRS = "EPSG:26916"
ANALYSIS_RES_M = 0.6

# Native ground sample distance per year. Inference runs at the native
# resolution and only the output is warped, so the model sees the input
# statistics it was trained on.
NATIVE_RES_M = {2012: 1.0, 2014: 1.0, 2016: 0.6, 2018: 0.6, 2020: 0.6, 2022: 0.6}

DRIVE_FOLDER = "SantaClara"
NAIPCHM_REPO = Path("/content/naip-chm")
MODEL_CHECKPOINT = "model/model_20251016.pt"
MODEL_CONFIG = "configs/config.yaml"
CHIP_SIZE = 432
CHIP_OVERLAP = 0.2

print(f"{SITE}: {len(YEARS)} dates, source={SOURCE}")

SantaClara: 6 dates, source=pc


## 2. Check the runtime

Fail here rather than three hours into a CPU run.

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or
      "No GPU detected. Set Runtime > Change runtime type > T4 GPU and rerun.")

GPU 0: Tesla T4 (UUID: GPU-21125375-4cb2-a48c-2022-c1c8dda884ff)



## 3. Install

Clone the canopy height model and install its dependencies. The model weights,
about 88 MB, ship inside the repository, so there is nothing else to download
for the network itself.

**After this cell finishes, restart the runtime** (Runtime > Restart session),
then run cell 1 again and continue from cell 4. The install replaces Colab's
preinstalled PyTorch, and continuing without a restart leaves a half-loaded
CUDA extension that fails in confusing ways at inference time.

In [13]:
%%bash
set -e
if [ ! -d /content/naip-chm ]; then
  git clone --depth 1 https://github.com/smorf-ntsg/naip-chm.git /content/naip-chm
fi
cd /content/naip-chm
pip install -q -r requirements.txt
pip install -q rio-cogeo pystac-client planetary-computer geopandas
echo
echo "Installed. Now restart the runtime, rerun cell 1, and continue from cell 4."


Installed. Now restart the runtime, rerun cell 1, and continue from cell 4.


Optionally install this project's library too, which brings in the stand
metric code and the fetch helper used below. Without it the notebook falls back
to an inline copy of the same two functions.

In [ ]:
%%bash
set -e
if [ ! -d /content/csdv ]; then
  echo "Clone or copy the CSDV repository to /content/csdv to use csdv_core here."
else
  pip install -q -e /content/csdv
fi

Clone or copy the CSDV repository to /content/csdv to use csdv_core here.


## 4. Mount Drive and lay out the cache

The conditioning rasters are about 1.65 GB and the NAIP mosaics are a few
hundred megabytes each. Caching them on Drive means a disconnected session
resumes instead of starting over.

In [2]:
import os, shutil
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER
DRIVE_COND_ROOT = Path("/content/drive/MyDrive/CSDV_Elkinsville")
DRIVE_COND = DRIVE_COND_ROOT / "conditioning_data"
DRIVE_NAIP = DRIVE_ROOT / "naip_mosaic"
DRIVE_CHM_RAW = DRIVE_ROOT / "chm_raw"
DRIVE_CHM_GRID = DRIVE_ROOT / "chm_grid"
for d in (DRIVE_COND, DRIVE_NAIP, DRIVE_CHM_RAW, DRIVE_CHM_GRID):
    d.mkdir(parents=True, exist_ok=True)

# Point the repo's conditioning directory at Drive so the download happens once.
local_cond = NAIPCHM_REPO / "data" / "conditioning_data"
local_cond.parent.mkdir(parents=True, exist_ok=True)
if local_cond.is_symlink():
    local_cond.unlink()
elif local_cond.exists():
    shutil.rmtree(local_cond)
os.symlink(DRIVE_COND, local_cond)

LOCAL_NAIP = Path("/content/work/naip"); LOCAL_NAIP.mkdir(parents=True, exist_ok=True)
LOCAL_CHM = Path("/content/work/chm"); LOCAL_CHM.mkdir(parents=True, exist_ok=True)
print("Drive cache:", DRIVE_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive cache: /content/drive/MyDrive/SantaClara


## Unzip the data once

In [ ]:
import zipfile

# Find all .zip files in the specific folder
for zip_path in DRIVE_ROOT.glob("*.ZIP"):
    # Create a matching folder name (e.g., 'data.zip' extracts to 'data' folder)
    extract_folder = DRIVE_ROOT / zip_path.stem

    # Open and extract the zip archive
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)
        print(f"Extracted: {zip_path.name} -> {extract_folder.name}/")


Extracted: m_3411948_ne_11_h_20160719.ZIP -> m_3411948_ne_11_h_20160719/
Extracted: n_3411948_ne_11_1_20050607.ZIP -> n_3411948_ne_11_1_20050607/
Extracted: m_3411948_ne_11_1_20120506.ZIP -> m_3411948_ne_11_1_20120506/
Extracted: m_3411948_ne_11_1_20140531.ZIP -> m_3411948_ne_11_1_20140531/
Extracted: m_3411948_ne_11_1_20100618.ZIP -> m_3411948_ne_11_1_20100618/
Extracted: m_3411948_ne_11_060_20180724.ZIP -> m_3411948_ne_11_060_20180724/
Extracted: m_3411948_ne_11_060_20200521.ZIP -> m_3411948_ne_11_060_20200521/
Extracted: m_3411948_ne_11_060_20240728.ZIP -> m_3411948_ne_11_060_20240728/
Extracted: m_3411948_ne_11_060_20220514.ZIP -> m_3411948_ne_11_060_20220514/
Extracted: m_3411948_ne_11_1_20090618.ZIP -> m_3411948_ne_11_1_20090618/


## 5. Conditioning rasters

Five static layers the model conditions on: elevation, a climate principal
component stack, a soil principal component stack, land cover and ecoregion.
Downloaded once, then read from the Drive cache.

In [3]:
REQUIRED = ["elevation.tif", "climate_pca.tif", "soil_pca.tif", "nlcd.tif", "ecoregion.tif"]

missing = [name for name in REQUIRED if not (DRIVE_COND / name).exists()]
if missing:
    print("Downloading:", missing)
    !cd {NAIPCHM_REPO} && echo n | python scripts/download_conditioning_data.py
else:
    print("Conditioning rasters already cached.")

still_missing = [name for name in REQUIRED if not (DRIVE_COND / name).exists()]
assert not still_missing, f"Conditioning rasters missing: {still_missing}"
for name in REQUIRED:
    print(f"  {name:16s} {(DRIVE_COND / name).stat().st_size / 1e6:8.1f} MB")

Conditioning rasters already cached.
  elevation.tif       236.0 MB
  climate_pca.tif     749.1 MB
  soil_pca.tif        733.6 MB
  nlcd.tif              8.3 MB
  ecoregion.tif         0.9 MB


## 6. Fetch NAIP

Each year's quads are read through a warped virtual raster already targeted at
that year's grid, so the mosaic step is a paste and every pixel is resampled
once. Access tokens expire after about an hour, so the catalogue is searched
inside the loop rather than once up front.

Filenames carry an eight-digit date because the model reads day of year from
the filename. Where a year's quads span more than one flight the median date is
used for the whole mosaic, and every individual date goes into the manifest.

In [ ]:
naip_paths = {}
naip_dates = {}
# .rglob("*.zip") searches the main directory AND all subdirectories
for image_path in DRIVE_NAIP.rglob("*.tif"):
  filename = str(image_path).split('/')[-1]
  date = filename.split('_')[-1].split('.')[0]
  year = date[0:4]
  naip_paths[year] = image_path
  naip_dates[year] = date
  print(f"{year}: {filename}")


2016: m_3411948_ne_11_h_20160719.tif
2005: n_3411948_ne_11_1_20050607.tif
2012: m_3411948_ne_11_1_20120506.tif
2014: m_3411948_ne_11_1_20140531.tif
2010: m_3411948_ne_11_1_20100618.tif
2018: m_3411948_ne_11_060_20180724.tif
2020: m_3411948_ne_11_060_20200521.tif
2024: m_3411948_ne_11_060_20240728.tif
2022: m_3411948_ne_11_060_20220514.tif
2009: m_3411948_ne_11_1_20090618.tif


In [ ]:
YEARS = list(naip_dates.keys())

In [ ]:
import rasterio
for year in YEARS:
    filepath = str(naip_paths[year])
    with rasterio.open(filepath) as src:
      print(f'{year}: {src.count} bands')

2016: 4 bands
2012: 4 bands
2014: 4 bands
2010: 4 bands
2018: 4 bands
2020: 4 bands
2024: 4 bands
2022: 4 bands
2009: 4 bands


In [ ]:
YEARS = [item for item in YEARS if item != "2005"]


## 7. Canopy height inference

One run per date. The model chips the image at 432 pixels with 20 percent
overlap, samples the conditioning rasters at each chip centre, and blends the
chips back together. Output is a cloud-optimised GeoTIFF of canopy height in
centimetres.

If a run fails on the mosaic size, the fallback is to split `INFER_BOUNDS` into
two by two sub-tiles with 150 m of overlap, infer each, and mosaic the
interiors. The overlap has to exceed the roughly 50 m unreliable rim, or the
seams carry the artefact.

In [ ]:
import subprocess, time

chm_raw = {}
for year in YEARS:
    tag = naip_dates[year]
    out_dir = LOCAL_CHM / f"{SITE}_{tag}"
    cached = list(DRIVE_CHM_RAW.glob(f"*{tag}*_chm.tif"))
    if cached:
        chm_raw[year] = cached[0]
        print(f"{year}: cached {cached[0].name}")
        continue

    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        "python", "scripts/inference.py",
        "--naip-quad", str(naip_paths[year]),
        "--output-dir", str(out_dir),
        "--model-checkpoint", MODEL_CHECKPOINT,
        "--config", MODEL_CONFIG,
        "--static-rasters-dir", "data/conditioning_data",
        "--chip-size", str(CHIP_SIZE),
        "--chip-overlap", f"{CHIP_OVERLAP:g}",
    ]
    started = time.time()
    result = subprocess.run(cmd, cwd=NAIPCHM_REPO, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-2000:]); print(result.stderr[-4000:])
        raise RuntimeError(f"Inference failed for {year}")

    produced = sorted(out_dir.glob("*_chm.tif"))[-1]
    shutil.copy2(produced, DRIVE_CHM_RAW / produced.name)
    chm_raw[year] = produced
    print(f"{year}: {produced.name} in {time.time() - started:.0f} s")

2016: cached m_3411948_ne_11_h_20160719_chm.tif
2012: m_3411948_ne_11_1_20120506_chm.tif in 145 s
2014: m_3411948_ne_11_1_20140531_chm.tif in 152 s
2010: m_3411948_ne_11_1_20100618_chm.tif in 154 s
2018: m_3411948_ne_11_060_20180724_chm.tif in 141 s
2020: m_3411948_ne_11_060_20200521_chm.tif in 149 s
2024: m_3411948_ne_11_060_20240728_chm.tif in 147 s
2022: m_3411948_ne_11_060_20220514_chm.tif in 146 s
2009: m_3411948_ne_11_1_20090618_chm.tif in 158 s


In [1]:
DRIVE_CHM_RAW

NameError: name 'DRIVE_CHM_RAW' is not defined

## Test SID for 2024

In [7]:
sid_path = DRIVE_ROOT / "mrsid_test/ortho_1-1_hm_s_ca111_2024_1.sid"
chm_2024_path = DRIVE_CHM_RAW / "m_3411948_ne_11_060_20240728_chm.tif"
sid_tif_path = DRIVE_ROOT / "mrsid_test/m_3411948_ne_11_060_20240728.tif"
# Clip SID to 2024 DOQQ extent

In [ ]:
import subprocess
import time
import rasterio
import matplotlib.pyplot as plt
import numpy as np

# 1. Run Inference
out_dir_sid = LOCAL_CHM / f"{SITE}_2024_sid_test"
out_dir_sid.mkdir(parents=True, exist_ok=True)

print("Starting inference on the converted TIF...")
cmd = [
    "python", "scripts/inference.py",
    "--naip-quad", str(sid_tif_path),
    "--output-dir", str(out_dir_sid),
    "--model-checkpoint", MODEL_CHECKPOINT,
    "--config", MODEL_CONFIG,
    "--static-rasters-dir", "data/conditioning_data",
    "--chip-size", str(CHIP_SIZE),
    "--chip-overlap", f"{CHIP_OVERLAP:g}",
]

started = time.time()
result = subprocess.run(cmd, cwd=NAIPCHM_REPO, capture_output=True, text=True)

if result.returncode != 0:
    print(result.stdout[-2000:])
    print(result.stderr[-4000:])
    raise RuntimeError("Inference failed for the SID-converted TIF")

produced_sid_chm = sorted(out_dir_sid.glob("*_chm.tif"))[-1]
print(f"Inference completed in {time.time() - started:.0f} s. Output: {produced_sid_chm.name}")

# 2. Compare the CHMs
print("Generating comparison plots...")
with rasterio.open(chm_2024_path) as src1, rasterio.open(produced_sid_chm) as src2:
    data1 = src1.read(1)
    data2 = src2.read(1)

    # Match shapes just in case there are slight differences in extent padding
    min_h = min(data1.shape[0], data2.shape[0])
    min_w = min(data1.shape[1], data2.shape[1])
    data1 = data1[:min_h, :min_w]
    data2 = data2[:min_h, :min_w]

# Filter out nodata and NaNs
mask = (data1 > -9999) & (data2 > -9999) & (~np.isnan(data1)) & (~np.isnan(data2))
d1_valid = data1[mask]
d2_valid = data2[mask]

# Subsample for the scatterplot
sample_size = min(100000, len(d1_valid))
idx = np.random.choice(len(d1_valid), sample_size, replace=False)
d1_samp = d1_valid[idx]
d2_samp = d2_valid[idx]

# 3. Plotting
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

vmax_val = np.percentile(d1_valid, 99) if len(d1_valid) > 0 else 3500

# Map 1
im1 = axes[0].imshow(data1, cmap='viridis', vmin=0, vmax=vmax_val)
axes[0].set_title('Original 2024 CHM')
plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
axes[0].axis('off')

# Map 2
im2 = axes[1].imshow(data2, cmap='viridis', vmin=0, vmax=vmax_val)
axes[1].set_title('SID-converted 2024 CHM')
plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
axes[1].axis('off')

# Scatter
axes[2].scatter(d1_samp, d2_samp, alpha=0.1, s=1, c='blue')
# Add a 1:1 line for reference
max_val = max(np.max(d1_samp), np.max(d2_samp)) if len(d1_samp) > 0 else 3500
axes[2].plot([0, max_val], [0, max_val], 'r--', label='1:1 Line')
axes[2].set_xlabel('Original CHM (cm)')
axes[2].set_ylabel('SID-converted CHM (cm)')
axes[2].set_title('Pixel-wise Comparison')
axes[2].legend()

plt.tight_layout()
plt.show()


Starting inference on the converted TIF...
Inference completed in 173 s. Output: m_3411948_ne_11_060_20240728_chm.tif
Generating comparison plots...


In [ ]:
import subprocess
import time
import rasterio
import matplotlib.pyplot as plt
import numpy as np

# 1. Run Inference
out_dir_sid = LOCAL_CHM / f"{SITE}_2024_sid_test"
# out_dir_sid.mkdir(parents=True, exist_ok=True)
sid_path = DRIVE_ROOT / "mrsid_test/ortho_1-1_hm_s_ca111_2024_1.sid"
chm_2024_path = DRIVE_CHM_RAW / "m_3411948_ne_11_060_20240728_chm.tif"
sid_tif_path = DRIVE_ROOT / "mrsid_test/m_3411948_ne_11_060_20240728.tif"
# Clip SID to 2024 DOQQ extent


# produced_sid_chm = sorted(out_dir_sid.glob("*_chm.tif"))[-1]
produced_sid_chm = DRIVE_CHM_RAW / "m_3411948_ne_11_060_20240728_chm_mrsid.tif"

# shutil.copy2(produced_sid_chm, DRIVE_CHM_RAW / "m_3411948_ne_11_060_20240728_chm_mrsid.tif")

# 2. Compare the CHMs
print("Generating comparison plots...")
with rasterio.open(chm_2024_path) as src1:
    data1 = src1.read(1)
    # Match shapes just in case there are slight differences in extent padding
print("read 1")
with rasterio.open(produced_sid_chm) as src2:
    data2 = src2.read(1)
print("read 2")
min_h = min(data1.shape[0], data2.shape[0])
min_w = min(data1.shape[1], data2.shape[1])

print(min_h)


data1 = data1[:min_h, :min_w]
data2 = data2[:min_h, :min_w]

# Filter out nodata and NaNs
print("Filtering and masking...")
mask = (data1 > -9999) & (data2 > -9999) & (~np.isnan(data1)) & (~np.isnan(data2))
d1_valid = data1[mask]
d2_valid = data2[mask]

# Subsample for the scatterplot
sample_size = min(100000, len(d1_valid))
idx = np.random.choice(len(d1_valid), sample_size, replace=False)
d1_samp = d1_valid[idx]
d2_samp = d2_valid[idx]
print("Subsampled...")

# 3. Plotting
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

vmax_val = np.percentile(d1_valid, 99) if len(d1_valid) > 0 else 3500

# Map 1
im1 = axes[0].imshow(data1, cmap='viridis', vmin=0, vmax=vmax_val)
axes[0].set_title('Original 2024 CHM')
plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
axes[0].axis('off')

# Map 2
im2 = axes[1].imshow(data2, cmap='viridis', vmin=0, vmax=vmax_val)
axes[1].set_title('SID-converted 2024 CHM')
plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
axes[1].axis('off')

# Scatter
axes[2].scatter(d1_samp, d2_samp, alpha=0.1, s=1, c='blue')
# Add a 1:1 line for reference
max_val = max(np.max(d1_samp), np.max(d2_samp)) if len(d1_samp) > 0 else 3500
axes[2].plot([0, max_val], [0, max_val], 'r--', label='1:1 Line')
axes[2].set_xlabel('Original CHM (cm)')
axes[2].set_ylabel('SID-converted CHM (cm)')
axes[2].set_title('Pixel-wise Comparison')
axes[2].legend()

plt.tight_layout()
plt.show()


Generating comparison plots...
read 1


In [13]:
%%bash
echo "GDAL Version installed in Colab:"
gdalinfo --version

echo -e "\nSearching for MrSID driver in GDAL formats..."
gdalinfo --formats | grep -i "mrsid" || echo "❌ MrSID driver NOT found in this GDAL build."

GDAL Version installed in Colab:

Searching for MrSID driver in GDAL formats...
❌ MrSID driver NOT found in this GDAL build.


bash: line 2: gdalinfo: command not found
bash: line 5: gdalinfo: command not found


## Predict NIR from RGB (untested)

In [ ]:
import rasterio
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from pathlib import Path
import os

# Configuration
train_year = '2012' # Use a 4-band image for training
target_year = '2005' # The 3-band image to augment
num_samples = 50000 # Number of random pixels to sample for training

train_path = naip_paths[train_year]
target_path = naip_paths[target_year] # Assuming this was kept in the dictionary earlier, or define it directly
if target_year not in naip_paths:
    # Manually search for the 2005 image if it was removed from the dictionary
    for p in DRIVE_NAIP.rglob("*.tif"):
        if "2005" in str(p):
            target_path = p
            break

print(f"Training Random Forest model using {train_year} image...")

# 1. Sample data from a 4-band image
with rasterio.open(train_path) as src:
    # Read all bands (assuming 1=R, 2=G, 3=B, 4=NIR)
    img_data = src.read()

    # Flatten and transpose to (num_pixels, 4)
    flat_data = img_data.reshape(4, -1).T

    # Remove nodata pixels if any (assuming 0 is nodata for simplicity)
    valid_mask = (flat_data[:, 0] > 0)
    valid_data = flat_data[valid_mask]

    # Randomly sample pixels for training
    indices = np.random.choice(valid_data.shape[0], min(num_samples, valid_data.shape[0]), replace=False)
    sampled_data = valid_data[indices]

    X_train = sampled_data[:, :3] # RGB
    y_train = sampled_data[:, 3]  # NIR

# 2. Train the Random Forest Model
rf = RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
print(f"Model trained. R^2 score on training subset: {rf.score(X_train, y_train):.3f}")

# 3. Apply the model to the 2005 3-band image
print(f"Applying model to {target_year} image...")
with rasterio.open(target_path) as src:
    meta = src.meta.copy()
    rgb_data = src.read() # Shape: (3, H, W)

    bands, height, width = rgb_data.shape

    # Reshape for prediction
    rgb_flat = rgb_data.reshape(3, -1).T

    # Predict NIR band (process in chunks if memory is an issue, doing all at once here)
    print("Predicting synthetic NIR band...")
    nir_flat = rf.predict(rgb_flat)
    nir_data = nir_flat.reshape(1, height, width).astype(rgb_data.dtype)

    # Combine RGB and synthetic NIR
    synthetic_4band = np.concatenate((rgb_data, nir_data), axis=0)

    # Update metadata for 4 bands
    meta.update(count=4)

    # Save the new 4-band image
    out_path = target_path.parent / f"{target_path.stem}_synthetic_4band.tif"
    with rasterio.open(out_path, 'w', **meta) as dst:
        dst.write(synthetic_4band)

print(f"Saved synthetic 4-band image to {out_path}")
# Update naip_paths to use the new image so inference works
naip_paths['2005'] = out_path
if '2005' not in YEARS:
    YEARS.append('2005')


## Below is old

In [ ]:
# Helper code to download from Files.com REST API response
import requests
import xml.etree.ElementTree as ET
import os

# 1. Paste the initial XML URL you found here
xml_url = "PASTE_YOUR_INITIAL_XML_URL_HERE"

# 2. Define your Google Drive destination path
destination_dir = "/content/drive/MyDrive/FilesCom_Downloads"
os.makedirs(destination_dir, exist_ok=True)
output_file = os.path.join(destination_dir, "downloaded_file.ext") # Change extension as needed

try:
    # Fetch the XML structure from the server
    response = requests.get(xml_url)
    response.raise_for_status()

    # Parse the XML string to find the true download link
    root = ET.fromstring(response.content)
    download_uri = root.find('.//download_uri').text

    print("🎯 Successfully extracted the direct URI! Streaming file to Google Drive...")

    # Use Bash/wget cleanly via Python's system execution to run the download
    os.system(f'wget -O "{output_file}" "{download_uri}"')
    print(f"🎉 Done! File saved to: {output_file}")

except Exception as e:
    print(f"❌ Automation failed: {e}")
    print("Ensure the XML URL hasn't expired or changed.")
